# Week 62, Day 5: Pairwise maximum entropy models of neural populations

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w062/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import itertools
import matplotlib.pyplot as plt
N = 10
W = np.array(list(itertools.product((0, 1), repeat=N)), dtype=float)       # all 1,024 words

def model_probs(h, J):
    E = W @ h + 0.5 * np.einsum("ki,ij,kj->k", W, J, W)
    E -= E.max()
    p = np.exp(E)
    return p / p.sum()

g0 = np.random.default_rng(625)
h_true = g0.uniform(-3.0, -2.2, N)
J_true = np.triu(g0.normal(0.25, 0.25, (N, N)), 1); J_true = J_true + J_true.T
p_true = model_probs(h_true, J_true)
idx = g0.choice(len(W), size=30000, p=p_true)
words = W[idx]
counts = np.bincount(idx, minlength=len(W))
p_emp = counts / counts.sum()
rates = words.mean(0)
C = np.corrcoef(words.T)
print(f"firing probability per bin: {rates.min():.3f} to {rates.max():.3f}")
print(f"pairwise correlations: mean {C[np.triu_indices(N, 1)].mean():.3f}, range {C[np.triu_indices(N, 1)].min():.3f} to {C[np.triu_indices(N, 1)].max():.3f}")

### Exercise: Fitting the pairwise model

Complete the gradient ascent: compute the model's first and second moments from its word probabilities, and step h and J along the differences from the data's moments (keep J symmetric with zero diagonal).

In [ ]:
M2 = words.T @ words / len(words)                    # data second moments <s_i s_j>
h_fit = np.log(rates / (1 - rates)); J_fit = np.zeros((N, N))
for it in range(3000):
    p = model_probs(h_fit, J_fit)
    m1 = W.T @ p
    m2 = ...
    h_fit += 1.0 * (rates - m1)
    dJ = 1.0 * (M2 - m2); np.fill_diagonal(dJ, 0)
    J_fit += ...
p_pair = model_probs(h_fit, J_fit)
p_ind = np.prod(np.where(W == 1, rates, 1 - rates), axis=1)
print(f"largest moment errors: rates {np.abs(W.T @ p_pair - rates).max():.5f}, pairs {np.abs((W * p_pair[:, None]).T @ W - M2).max():.5f}")
print(f"couplings: fitted against true, correlation {np.corrcoef(J_fit[np.triu_indices(N, 1)], J_true[np.triu_indices(N, 1)])[0, 1]:.3f}")

In [ ]:
import numpy as np
def test_moments():
    "the fitted model reproduces the data's firing rates and pairwise products to within 0.002"
    p = model_probs(h_fit, J_fit)
    assert np.abs(W.T @ p - rates).max() < 2e-3 and np.abs((W * p[:, None]).T @ W - M2).max() < 2e-3

test_moments()
print("All checks passed")

### Exercise: Patterns, synchrony and multi-information

Compute the probability of 6 or more neurons firing in the same bin under the data, the independent model and the pairwise model. Then compute the entropies S1 (independent), S2 (pairwise) and SN (the true model, which the simulation lets you know), and the fraction of multi-information captured.

In [ ]:
k_on = W.sum(1)
sync_emp, sync_ind, sync_pair = (p[k_on >= 6].sum() for p in (p_emp, p_ind, p_pair))
H = lambda p: -(p[p > 0] * np.log2(p[p > 0])).sum()
S1, S2, SN = H(p_ind), H(p_pair), H(p_true)
frac = ...
print(f"P(6 or more together): data {sync_emp:.2e}, independent {sync_ind:.2e}, pairwise {sync_pair:.2e}")
print(f"entropies (bits): independent {S1:.4f}, pairwise {S2:.4f}, true {SN:.4f}; fraction of multi-information captured {frac:.3f}")
plt.loglog(p_emp[p_emp > 0], p_ind[p_emp > 0], ".", ms=4, label="independent")
plt.loglog(p_emp[p_emp > 0], p_pair[p_emp > 0], ".", ms=4, label="pairwise")
plt.plot([1e-5, 1], [1e-5, 1], "k", lw=0.8); plt.xlabel("observed frequency of a word"); plt.ylabel("predicted"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_sync():
    "the independent model underestimates the probability of 6 or more neurons firing together by more than tenfold; the pairwise model is within a factor of 2"
    assert sync_emp / sync_ind > 10 and 0.5 < sync_emp / sync_pair < 2
def test_fraction():
    "the pairwise model captures more than 80% of the multi-information"
    assert frac > 0.8

test_sync()
test_fraction()
print("All checks passed")

### Exercise: Fitting 25 neurons with sampled moments

For 25 neurons, data come from a known pairwise model sampled by long Gibbs runs. Complete the learning loop: after a few Gibbs sweeps of the persistent chains, update h and J along the differences between data and chain moments. Then compare the pairwise correlations of the final chains with those of the data.

In [ ]:
def gibbs_sweeps(S, h, J, sweeps, g):
    for _ in range(sweeps):
        for i in range(S.shape[1]):
            field = h[i] + S @ J[:, i]
            S[:, i] = (g.random(len(S)) < 1 / (1 + np.exp(-field))).astype(float)
    return S

gb = np.random.default_rng(6251)
N25 = 25
hb = gb.uniform(-2.8, -2.0, N25)
Jb = np.triu(gb.normal(0.15, 0.2, (N25, N25)), 1); Jb = Jb + Jb.T
data25 = gibbs_sweeps((gb.random((8000, N25)) < 0.1).astype(float), hb, Jb, 60, gb)
r_d = data25.mean(0); M_d = data25.T @ data25 / len(data25)

h_l = np.log(r_d / (1 - r_d)); J_l = np.zeros((N25, N25))
chains = (gb.random((3000, N25)) < r_d).astype(float)
for it in range(150):
    chains = gibbs_sweeps(chains, h_l, J_l, 2, gb)
    r_m = chains.mean(0); M_m = chains.T @ chains / len(chains)
    h_l += ...
    dJ = 0.5 * (M_d - M_m); np.fill_diagonal(dJ, 0)
    J_l += dJ
chains = gibbs_sweeps(chains, h_l, J_l, 20, gb)
iu = np.triu_indices(N25, 1)
corr_data = np.corrcoef(data25.T)[iu]
corr_model = np.corrcoef(chains.T)[iu]
r_fit = np.corrcoef(corr_data, corr_model)[0, 1]
print(f"pairwise correlations, model against data: r = {r_fit:.3f}; mean data correlation {corr_data.mean():.3f}, model {corr_model.mean():.3f}")

In [ ]:
import numpy as np
def test_corr():
    "after Boltzmann learning, the model's pairwise correlations track the data's (correlation above 0.85)"
    assert r_fit > 0.85

test_corr()
print("All checks passed")